In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import akshare as ak
from tqdm import tqdm
from datetime import datetime, timedelta
from collections import Counter
from quant_utils import midterm_entry_signal_backtrack,backtest_single_stock


In [70]:
stock_list = ak.stock_info_a_code_name()

#测试版
stock_list = stock_list.head(100)

In [ ]:
#单只股票的测试

code = "000001"

df = ak.stock_zh_a_hist(
    symbol=code,
    period="daily",
    adjust="qfq",
)
df['MA20'] = df['收盘'].rolling(20).mean()
df['MA60'] = df['收盘'].rolling(60).mean()

rets = backtest_single_stock(df)

print("信号次数:", len(rets))
print("平均收益:", np.mean(rets))
print("胜率:", np.mean(np.array(rets) > 0))

信号次数: 85
平均收益: 0.12217076483227493
胜率: 0.32941176470588235


In [73]:
df 

,日期,股票代码,开盘,收盘,最高,最低,成交量,成交额,振幅,涨跌幅,涨跌额,换手率,MA20,MA60
0,1991-04-03,000001,-2.72,-2.72,-2.72,-2.72,1,5.000000e+03,0.00,2.86,0.08,0.00,NaN,NaN
1,1991-04-04,000001,-2.73,-2.73,-2.73,-2.73,3,1.500000e+04,0.00,-0.37,-0.01,0.00,NaN,NaN
2,1991-04-05,000001,-2.73,-2.73,-2.73,-2.73,2,1.000000e+04,0.00,0.00,0.00,0.00,NaN,NaN
3,1991-04-06,000001,-2.73,-2.73,-2.73,-2.73,7,3.400000e+04,0.00,0.00,0.00,0.00,NaN,NaN
4,1991-04-08,000001,-2.73,-2.73,-2.73,-2.73,2,1.000000e+04,0.00,0.00,0.00,0.00,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8334,2026-02-09,000001,11.05,11.07,11.11,11.01,619717,6.852182e+08,0.90,0.18,0.02,0.32,11.0575,11.400333
8335,2026-02-10,000001,11.07,11.06,11.10,11.02,600430,6.641402e+08,0.72,-0.09,-0.01,0.31,11.0370,11.388833
8336,2026-02-11,000001,11.06,11.07,11.09,11.02,431041,4.768019e+08,0.63,0.09,0.01,0.22,11.0225,11.378833
8337,2026-02-12,000001,11.07,10.96,11.08,10.93,681340,7.483385e+08,1.36,-0.99,-0.11,0.35,11.0050,11.368333


In [ ]:
all_rets = []
stock_results = []

for _, row in tqdm(stock_list.iterrows(), total=len(stock_list)):
    code = row['code']
    
    try:
        df = ak.stock_zh_a_hist(
            symbol=code,
            period="daily",
            adjust="qfq",
        )
        
        if df.shape[0] < 100:
            continue
        
        df['MA20'] = df['收盘'].rolling(20).mean()
        df['MA60'] = df['收盘'].rolling(60).mean()

        rets = backtest_single_stock(df)

        if len(rets) == 0:
            continue

        all_rets.extend(rets)

        stock_results.append({
            "code": code,
            "信号次数": len(rets),
            "平均收益": np.mean(rets),
            "胜率": np.mean(np.array(rets) > 0)
        })

    except:
        continue


# ===== 整体统计 =====
print("总信号次数:", len(all_rets))
print("整体平均收益:", np.mean(all_rets))
print("整体胜率:", np.mean(np.array(all_rets) > 0))

100%|██████████| 100/100 [06:48<00:00,  4.08s/it]

总信号次数: 5550
整体平均收益: 0.0780200209940097
整体胜率: 0.3396396396396396


In [75]:
all_rets = [r for r in all_rets if abs(r) < 1]
wins = [r for r in all_rets if r > 0]
losses = [r for r in all_rets if r <= 0]

print("平均盈利:", np.mean(wins))
print("平均亏损:", np.mean(losses))
print("盈亏比:", abs(np.mean(wins) / np.mean(losses)))

平均盈利: 0.16710665843561986
平均亏损: -0.06740422435943444
盈亏比: 2.479171892024454


In [ ]:
print("最大盈利:", np.max(all_rets))
print("最大亏损:", np.min(all_rets))